# TFM — Construcción y análisis de un modelo de Expected Goals (xG) en el fútbol de élite
## Sección 4: Análisis Exploratorio de Datos (EDA)

**Autor:** Niklas  
**Fuente de datos:** StatsBomb Open Data (github.com/statsbomb/open-data)  
**Competiciones:** La Liga (2004/05–2020/21), UEFA Champions League, Copa del Mundo FIFA 2018 y 2022

---

Este notebook cubre el análisis exploratorio completo del dataset de disparos extraído de StatsBomb Open Data. El objetivo es entender la estructura de los datos, detectar valores ausentes y outliers, y visualizar las relaciones entre las variables que se usarán para construir el modelo xG.

**Índice:**
1. Instalación y carga de librerías
2. Descarga de datos desde StatsBomb
3. Construcción del dataset de disparos
4. Análisis de valores ausentes (missing values)
5. Estadísticas descriptivas
6. Distribución de la variable objetivo
7. Distribución de variables numéricas
8. Distribución de variables categóricas
9. Detección de outliers
10. Análisis de correlaciones
11. Comparativas por competición
12. Visualización espacial de disparos
13. Conclusiones del EDA

## 1. Instalación y carga de librerías

In [ ]:
# Instalación de librerías específicas de fútbol (solo necesario en Colab)
!pip install statsbombpy mplsoccer --quiet

In [ ]:
# ─── Librerías estándar de análisis de datos ───────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

# ─── StatsBomb ─────────────────────────────────────────────────────────────
from statsbombpy import sb

# ─── Visualización de campo de fútbol ──────────────────────────────────────
from mplsoccer import Pitch, VerticalPitch

# ─── Configuración general ─────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Estilo visual consistente con seaborn
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = {'gol': '#e63946', 'no_gol': '#457b9d', 'neutral': '#2E4057'}

print('✅ Librerías cargadas correctamente.')

## 2. Descarga de datos desde StatsBomb

Descargamos los datos de eventos para las tres competiciones seleccionadas:
- **La Liga** (competition_id=11): temporadas 2004/05 a 2020/21
- **UEFA Champions League** (competition_id=16): varias temporadas
- **Copa del Mundo FIFA** (competition_id=43): 2018 y 2022

> ⚠️ **Nota:** La descarga puede tardar varios minutos dependiendo de la conexión. Los datos se descargan directamente del repositorio público de StatsBomb en GitHub.

In [ ]:
# Ver todas las competiciones disponibles en StatsBomb Open Data
competitions = sb.competitions()
print(f'Total de competiciones disponibles: {len(competitions)}')
print('\nCompeticiones seleccionadas para el TFM:')
selected = competitions[
    competitions['competition_id'].isin([11, 16, 43])
][['competition_id', 'competition_name', 'season_name']]
print(selected.to_string(index=False))

In [ ]:
def cargar_disparos_competicion(competition_id, season_id, competition_name):
    """
    Carga todos los disparos de una temporada y competición dadas.
    Devuelve un DataFrame con solo los eventos de tipo 'Shot'.
    """
    try:
        matches = sb.matches(competition_id=competition_id, season_id=season_id)
        all_shots = []
        for match_id in matches['match_id']:
            events = sb.events(match_id=match_id)
            shots = events[events['type'] == 'Shot'].copy()
            if not shots.empty:
                shots['competition'] = competition_name
                shots['season_id'] = season_id
                shots['match_id'] = match_id
                all_shots.append(shots)
        if all_shots:
            return pd.concat(all_shots, ignore_index=True)
    except Exception as e:
        print(f'  Error en {competition_name} temporada {season_id}: {e}')
    return pd.DataFrame()

print('Función de carga definida ✅')

In [ ]:
# ─── Definición de competiciones y temporadas a descargar ──────────────────
#
# competition_id=11 → La Liga (España)
# competition_id=16 → UEFA Champions League
# competition_id=43 → Copa del Mundo FIFA
#
# Los season_id se obtienen del dataframe 'competitions' cargado arriba.
# Filtramos las disponibles en StatsBomb Open Data.

competiciones_a_cargar = competitions[
    competitions['competition_id'].isin([11, 16, 43])
][['competition_id', 'competition_name', 'season_id', 'season_name']].values.tolist()

print(f'Se descargarán {len(competiciones_a_cargar)} temporada(s)/competición(es).')
print('Iniciando descarga... (puede tardar varios minutos)\n')

all_shots_list = []
for comp_id, comp_name, seas_id, seas_name in competiciones_a_cargar:
    print(f'  → Descargando: {comp_name} — {seas_name}')
    shots_temp = cargar_disparos_competicion(comp_id, seas_id, comp_name)
    if not shots_temp.empty:
        all_shots_list.append(shots_temp)

# Concatenar todos los disparos en un único DataFrame
df_raw = pd.concat(all_shots_list, ignore_index=True)
print(f'\n✅ Descarga completada.')
print(f'   Total de disparos descargados: {len(df_raw):,}')
print(f'   Columnas disponibles: {df_raw.shape[1]}')

## 3. Construcción del dataset de disparos

Seleccionamos las variables relevantes para el modelo xG, calculamos las variables derivadas (distancia y ángulo al gol) y binarizamos la variable objetivo.

In [ ]:
# ─── Selección de columnas relevantes ──────────────────────────────────────
columnas = [
    'id', 'match_id', 'competition', 'season_id', 'minute', 'player',
    'team', 'location',
    'shot_outcome', 'shot_type', 'shot_body_part', 'shot_technique',
    'shot_statsbomb_xg', 'under_pressure', 'shot_first_time'
]

# Filtrar solo columnas que existan en el DataFrame
columnas_disponibles = [c for c in columnas if c in df_raw.columns]
df = df_raw[columnas_disponibles].copy()

# ─── Extraer coordenadas x, y desde la columna 'location' ──────────────────
# StatsBomb almacena la localización como lista [x, y]
df['x'] = df['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
df['y'] = df['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)
df.drop(columns=['location'], inplace=True)

# ─── Variable objetivo: gol (1) vs. no gol (0) ─────────────────────────────
# Excluimos penaltis (shot_type == 'Penalty') por su naturaleza atípica
df = df[df['shot_type'] != 'Penalty'].copy()
df['gol'] = (df['shot_outcome'] == 'Goal').astype(int)

# ─── Variables derivadas: distancia y ángulo al centro de la portería ──────
# Sistema de coordenadas StatsBomb: campo 120x80, portería rival en x=120, y entre 36 y 44
GOAL_X = 120
GOAL_Y = 40  # Centro de la portería
GOAL_POST_Y1 = 36
GOAL_POST_Y2 = 44

# Distancia euclidea al centro de la portería
df['shot_distance'] = np.sqrt((df['x'] - GOAL_X)**2 + (df['y'] - GOAL_Y)**2)

# Ángulo de disparo (en grados) — ángulo entre los dos postes desde la posición del disparo
def calcular_angulo(x, y):
    """Calcula el ángulo de visión de la portería desde la posición (x, y)."""
    a = np.array([x - GOAL_X, y - GOAL_POST_Y1])  # Vector al poste 1
    b = np.array([x - GOAL_X, y - GOAL_POST_Y2])  # Vector al poste 2
    cos_angle = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-6)
    cos_angle = np.clip(cos_angle, -1, 1)
    return np.degrees(np.arccos(cos_angle))

df['shot_angle'] = df.apply(lambda row: calcular_angulo(row['x'], row['y']), axis=1)

# ─── Limpieza de variables categóricas ─────────────────────────────────────
for col in ['shot_outcome', 'shot_type', 'shot_body_part', 'shot_technique']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Booleanas → 0/1
for col in ['under_pressure', 'shot_first_time']:
    if col in df.columns:
        df[col] = df[col].fillna(False).astype(int)

print(f'✅ Dataset de disparos construido.')
print(f'   Disparos totales (sin penaltis): {len(df):,}')
print(f'   Goles: {df["gol"].sum():,} ({df["gol"].mean()*100:.1f}%)')
print(f'   Columnas finales: {df.shape[1]}')
print(f'\nPrimeras filas:')
df.head(3)

## 4. Análisis de valores ausentes (Missing Values)

Siguiendo la metodología CRISP-DM estudiada en clase, el primer paso del análisis exploratorio es cuantificar y entender los valores ausentes antes de tomar decisiones de imputación o eliminación.

In [ ]:
# ─── Resumen de missing values ─────────────────────────────────────────────
missing = pd.DataFrame({
    'n_missing': df.isnull().sum(),
    'pct_missing': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('pct_missing', ascending=False)

missing_positivos = missing[missing['n_missing'] > 0]

print('Variables con valores ausentes:')
if missing_positivos.empty:
    print('  → No hay valores ausentes en el dataset de disparos.')
else:
    print(missing_positivos.to_string())

# ─── Gráfico de missing values ─────────────────────────────────────────────
if not missing_positivos.empty:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing_positivos) * 0.5)))
    bars = ax.barh(missing_positivos.index, missing_positivos['pct_missing'],
                   color=COLORS['neutral'], edgecolor='white')
    ax.set_xlabel('% de valores ausentes')
    ax.set_title('Porcentaje de valores ausentes por variable', fontweight='bold')
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='Umbral 50%')
    for bar, val in zip(bars, missing_positivos['pct_missing']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n💾 Gráfico guardado como missing_values.png')

## 5. Estadísticas descriptivas

In [ ]:
# ─── Estadísticas descriptivas de variables numéricas ─────────────────────
vars_numericas = ['shot_distance', 'shot_angle', 'minute', 'shot_statsbomb_xg', 'x', 'y']
vars_numericas = [v for v in vars_numericas if v in df.columns]

desc = df[vars_numericas].describe().T
desc['skewness'] = df[vars_numericas].skew().round(3)  # Asimetría (como en clase)
print('Estadísticas descriptivas de variables numéricas:')
desc.round(3)

In [ ]:
# ─── Resumen por competición ───────────────────────────────────────────────
resumen_comp = df.groupby('competition').agg(
    n_disparos=('gol', 'count'),
    n_goles=('gol', 'sum'),
    tasa_conversion=('gol', 'mean'),
    distancia_media=('shot_distance', 'mean'),
    angulo_medio=('shot_angle', 'mean'),
    xg_medio=('shot_statsbomb_xg', 'mean')
).round(3)
resumen_comp['tasa_conversion'] = (resumen_comp['tasa_conversion'] * 100).round(1)
print('Resumen por competición:')
resumen_comp

## 6. Distribución de la variable objetivo

Análisis del desbalance de clases — aspecto crítico en modelos de clasificación, estudiado en la asignatura de Machine Learning.

In [ ]:
# ─── Distribución de la variable objetivo (gol / no gol) ──────────────────
conteo_gol = df['gol'].value_counts()
pct_gol = df['gol'].value_counts(normalize=True) * 100

print('Distribución de la variable objetivo:')
print(f'  No gol (0): {conteo_gol[0]:,}  ({pct_gol[0]:.1f}%)')
print(f'  Gol    (1): {conteo_gol[1]:,}  ({pct_gol[1]:.1f}%)')
print(f'\n  → Ratio de desbalance: {conteo_gol[0]/conteo_gol[1]:.1f}:1')
print('  → Dataset fuertemente desbalanceado: esto se tendrá en cuenta en la modelización.')
print('     Se usará AUC-ROC como métrica principal (robusta al desbalance).')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Gráfico de barras
labels = ['No gol', 'Gol']
values = [conteo_gol[0], conteo_gol[1]]
bars = axes[0].bar(labels, values,
                   color=[COLORS['no_gol'], COLORS['gol']],
                   edgecolor='white', width=0.5)
for bar, val, pct in zip(bars, values, [pct_gol[0], pct_gol[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Distribución de la variable objetivo', fontweight='bold')
axes[0].set_ylabel('Número de disparos')
axes[0].set_ylim(0, max(values) * 1.15)

# Gráfico de tarta
axes[1].pie([conteo_gol[0], conteo_gol[1]],
            labels=['No gol', 'Gol'],
            colors=[COLORS['no_gol'], COLORS['gol']],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11})
axes[1].set_title('Proporción gol / no gol', fontweight='bold')

plt.tight_layout()
plt.savefig('distribucion_variable_objetivo.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como distribucion_variable_objetivo.png')

## 7. Distribución de variables numéricas

Análisis de las dos variables más importantes para el modelo xG: distancia y ángulo de disparo.

In [ ]:
# ─── Histogramas y boxplots de distancia y ángulo por resultado ────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

vars_plot = [
    ('shot_distance', 'Distancia al gol (metros equiv.)'),
    ('shot_angle',    'Ángulo de disparo (grados)')
]

for i, (var, label) in enumerate(vars_plot):
    if var not in df.columns:
        continue

    # Histograma
    for val, color, name in [(0, COLORS['no_gol'], 'No gol'), (1, COLORS['gol'], 'Gol')]:
        axes[i, 0].hist(df[df['gol'] == val][var].dropna(),
                        bins=40, alpha=0.6, color=color, label=name, density=True)
    axes[i, 0].set_xlabel(label)
    axes[i, 0].set_ylabel('Densidad')
    axes[i, 0].set_title(f'Distribución de {label}', fontweight='bold')
    axes[i, 0].legend()

    # Boxplot
    data_box = [df[df['gol'] == 0][var].dropna(), df[df['gol'] == 1][var].dropna()]
    bp = axes[i, 1].boxplot(data_box, labels=['No gol', 'Gol'],
                            patch_artist=True,
                            medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor(COLORS['no_gol'])
    bp['boxes'][1].set_facecolor(COLORS['gol'])
    for box in bp['boxes']:
        box.set_alpha(0.7)
    axes[i, 1].set_ylabel(label)
    axes[i, 1].set_title(f'Boxplot de {label} por resultado', fontweight='bold')

    # Añadir medias
    for j, val in enumerate([0, 1]):
        media = df[df['gol'] == val][var].mean()
        axes[i, 1].text(j+1, media, f'μ={media:.1f}', ha='center', va='bottom',
                        fontsize=9, color='darkred', fontweight='bold')

plt.tight_layout()
plt.savefig('distribucion_distancia_angulo.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como distribucion_distancia_angulo.png')

In [ ]:
# ─── Tasa de conversión por tramo de distancia ─────────────────────────────
df['distancia_tramo'] = pd.cut(df['shot_distance'],
                                bins=[0, 5, 10, 15, 20, 25, 30, 40, 60, 120],
                                labels=['0-5', '5-10', '10-15', '15-20',
                                        '20-25', '25-30', '30-40', '40-60', '>60'])

conv_distancia = df.groupby('distancia_tramo', observed=True)['gol'].agg(['mean', 'count']).reset_index()
conv_distancia.columns = ['tramo', 'tasa_conversion', 'n_disparos']
conv_distancia['tasa_conversion'] = conv_distancia['tasa_conversion'] * 100

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

bars = ax1.bar(conv_distancia['tramo'], conv_distancia['tasa_conversion'],
               color=COLORS['gol'], alpha=0.75, label='Tasa de conversión (%)')
ax2.plot(conv_distancia['tramo'], conv_distancia['n_disparos'],
         'o-', color=COLORS['neutral'], linewidth=2, markersize=6, label='N° disparos')

ax1.set_xlabel('Tramo de distancia (metros equiv.)')
ax1.set_ylabel('Tasa de conversión (%)', color=COLORS['gol'])
ax2.set_ylabel('Número de disparos', color=COLORS['neutral'])
ax1.set_title('Tasa de conversión y volumen de disparos por tramo de distancia',
              fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('conversion_por_distancia.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como conversion_por_distancia.png')

## 8. Distribución de variables categóricas

In [ ]:
# ─── Tasa de conversión por parte del cuerpo, técnica y situación ──────────
vars_cat = [
    ('shot_body_part', 'Parte del cuerpo'),
    ('shot_technique', 'Técnica de disparo'),
    ('under_pressure', 'Bajo presión defensiva'),
    ('shot_first_time', 'Disparo a la primera')
]
vars_cat = [(v, l) for v, l in vars_cat if v in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, (var, label) in enumerate(vars_cat):
    tabla = df.groupby(var)['gol'].agg(['mean', 'count']).reset_index()
    tabla.columns = [var, 'tasa_conversion', 'n_disparos']
    tabla['tasa_conversion'] = tabla['tasa_conversion'] * 100
    tabla = tabla.sort_values('tasa_conversion', ascending=True)

    bars = axes[i].barh(tabla[var].astype(str), tabla['tasa_conversion'],
                        color=COLORS['neutral'], alpha=0.8, edgecolor='white')
    for bar, n in zip(bars, tabla['n_disparos']):
        axes[i].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                     f'{bar.get_width():.1f}%  (n={n:,})',
                     va='center', fontsize=9)
    axes[i].set_xlabel('Tasa de conversión (%)')
    axes[i].set_title(f'Conversión por {label}', fontweight='bold')
    axes[i].set_xlim(0, tabla['tasa_conversion'].max() * 1.35)

plt.tight_layout()
plt.savefig('conversion_variables_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como conversion_variables_categoricas.png')

## 9. Detección de outliers

Utilizamos el método del rango intercuartílico (IQR), estudiado en la asignatura de Minería de Datos, por ser más robusto que el método de la desviación típica para distribuciones asimétricas.

In [ ]:
# ─── Detección de outliers por IQR ─────────────────────────────────────────
def detectar_outliers_iqr(serie, k=1.5):
    """Devuelve máscara booleana de outliers usando el método IQR."""
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return (serie < lower) | (serie > upper), lower, upper

print('Análisis de outliers (método IQR, k=1.5):')
print('=' * 60)
for var in ['shot_distance', 'shot_angle', 'minute']:
    if var not in df.columns:
        continue
    mask, lower, upper = detectar_outliers_iqr(df[var].dropna())
    n_out = mask.sum()
    pct_out = n_out / len(df) * 100
    print(f'\n{var}:')
    print(f'  Límite inferior: {lower:.2f}')
    print(f'  Límite superior: {upper:.2f}')
    print(f'  Outliers detectados: {n_out:,} ({pct_out:.2f}%)')
    print(f'  → Decisión: los outliers en distancia/ángulo son disparos lejanos')
    print(f'    legítimos (ej. disparos desde el propio campo). Se conservan.')

print('\n' + '=' * 60)
print('Nota: en el contexto de xG, no hay outliers en sentido estricto —')
print('todo disparo es un evento real y válido del partido.')

## 10. Análisis de correlaciones

In [ ]:
# ─── Matriz de correlaciones entre variables numéricas ─────────────────────
vars_corr = ['gol', 'shot_distance', 'shot_angle', 'minute',
             'shot_statsbomb_xg', 'under_pressure', 'shot_first_time']
vars_corr = [v for v in vars_corr if v in df.columns]

corr_matrix = df[vars_corr].corr().round(3)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Triángulo superior
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de correlaciones (variables numéricas)', fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como matriz_correlaciones.png')

print('\nCorrelaciones con la variable objetivo (gol):')
print(corr_matrix['gol'].drop('gol').sort_values(ascending=False).to_string())

## 11. Comparativas por competición

In [ ]:
# ─── Comparativa de distribuciones por competición ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metricas = [
    ('shot_distance', 'Distancia al gol'),
    ('shot_angle',    'Ángulo de disparo (°)'),
    ('shot_statsbomb_xg', 'xG StatsBomb')
]

for i, (var, label) in enumerate(metricas):
    if var not in df.columns:
        continue
    competiciones = df['competition'].unique()
    data_plot = [df[df['competition'] == c][var].dropna().values for c in competiciones]
    bp = axes[i].boxplot(data_plot,
                         labels=[c.replace('UEFA ', '').replace('FIFA ', '')[:12]
                                 for c in competiciones],
                         patch_artist=True,
                         medianprops={'color': 'black', 'linewidth': 2})
    colors_list = [COLORS['gol'], COLORS['no_gol'], COLORS['neutral']]
    for box, color in zip(bp['boxes'], colors_list):
        box.set_facecolor(color)
        box.set_alpha(0.7)
    axes[i].set_title(f'{label} por competición', fontweight='bold')
    axes[i].set_ylabel(label)
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('comparativa_por_competicion.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como comparativa_por_competicion.png')

In [ ]:
# ─── Evolución temporal de la tasa de conversión en La Liga ───────────────
if 'La Liga' in df['competition'].values and 'season_id' in df.columns:
    la_liga = df[df['competition'] == 'La Liga'].copy()
    evolucion = la_liga.groupby('season_id')['gol'].agg(['mean', 'count']).reset_index()
    evolucion.columns = ['season_id', 'tasa_conversion', 'n_disparos']
    evolucion['tasa_conversion'] = evolucion['tasa_conversion'] * 100
    evolucion = evolucion.sort_values('season_id')

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(range(len(evolucion)), evolucion['tasa_conversion'],
            'o-', color=COLORS['gol'], linewidth=2.5, markersize=7)
    ax.fill_between(range(len(evolucion)), evolucion['tasa_conversion'],
                    alpha=0.15, color=COLORS['gol'])
    ax.set_xticks(range(len(evolucion)))
    ax.set_xticklabels(evolucion['season_id'], rotation=45, ha='right')
    ax.set_ylabel('Tasa de conversión (%)')
    ax.set_title('Evolución de la tasa de conversión en La Liga por temporada',
                 fontweight='bold')
    ax.axhline(evolucion['tasa_conversion'].mean(), color='gray',
               linestyle='--', alpha=0.6, label=f"Media: {evolucion['tasa_conversion'].mean():.1f}%")
    ax.legend()
    plt.tight_layout()
    plt.savefig('evolucion_temporal_laliga.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('💾 Gráfico guardado como evolucion_temporal_laliga.png')

## 12. Visualización espacial de disparos

Mapa de calor de disparos sobre el campo de fútbol, diferenciando goles y no goles. Esta visualización es característica del análisis de datos de fútbol y es una de las más informativas para entender desde dónde se generan las ocasiones de gol.

In [ ]:
# ─── Mapa de disparos sobre el campo (mplsoccer) ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (resultado, titulo, color) in enumerate([
    (0, 'Disparos sin gol', COLORS['no_gol']),
    (1, 'Goles', COLORS['gol'])
]):
    subset = df[df['gol'] == resultado]
    pitch = VerticalPitch(pitch_type='statsbomb', pitch_color='grass',
                          line_color='white', half=True)
    pitch.draw(ax=axes[i])
    pitch.kdeplot(subset['x'], subset['y'], ax=axes[i],
                  fill=True, levels=100, cmap='Reds' if resultado == 1 else 'Blues',
                  alpha=0.7)
    axes[i].set_title(f'{titulo} (n={len(subset):,})', fontweight='bold', fontsize=12)

plt.suptitle('Distribución espacial de disparos', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('mapa_disparos_campo.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como mapa_disparos_campo.png')

In [ ]:
# ─── Scatter plot: distancia vs ángulo coloreado por resultado ─────────────
fig, ax = plt.subplots(figsize=(10, 6))

sample = df.sample(min(5000, len(df)), random_state=42)  # Muestra para no saturar
scatter = ax.scatter(
    sample['shot_distance'], sample['shot_angle'],
    c=sample['gol'],
    cmap='RdBu_r', alpha=0.4, s=15,
    vmin=0, vmax=1
)

legend_elements = [
    mpatches.Patch(facecolor=COLORS['no_gol'], alpha=0.6, label='No gol'),
    mpatches.Patch(facecolor=COLORS['gol'], alpha=0.6, label='Gol')
]
ax.legend(handles=legend_elements, loc='upper right')
ax.set_xlabel('Distancia al gol (metros equiv.)')
ax.set_ylabel('Ángulo de disparo (grados)')
ax.set_title('Relación entre distancia, ángulo y resultado del disparo',
             fontweight='bold')

plt.tight_layout()
plt.savefig('scatter_distancia_angulo_gol.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado como scatter_distancia_angulo_gol.png')

## 13. Conclusiones del EDA

Resumen de los hallazgos más relevantes del análisis exploratorio, que guiarán las decisiones de la fase de modelización.

In [ ]:
# ─── Resumen final del EDA ─────────────────────────────────────────────────
print('=' * 65)
print('RESUMEN DEL ANÁLISIS EXPLORATORIO DE DATOS (EDA)')
print('=' * 65)

print(f"""
1. VOLUMEN DE DATOS
   • Total de disparos (sin penaltis): {len(df):,}
   • Goles: {df['gol'].sum():,} ({df['gol'].mean()*100:.1f}%)
   • Dataset fuertemente desbalanceado (~{round(df['gol'].value_counts(normalize=True)[0]/df['gol'].value_counts(normalize=True)[1])}:1)
   → Se usará AUC-ROC como métrica principal de evaluación.

2. DISTANCIA Y ÁNGULO (variables más predictivas)
   • Distancia media de goles: {df[df['gol']==1]['shot_distance'].mean():.1f} (vs {df[df['gol']==0]['shot_distance'].mean():.1f} en no-goles)
   • Ángulo medio de goles: {df[df['gol']==1]['shot_angle'].mean():.1f}° (vs {df[df['gol']==0]['shot_angle'].mean():.1f}° en no-goles)
   → Clara separación entre goles y no goles en ambas variables.

3. VARIABLES CATEGÓRICAS
   → Parte del cuerpo y técnica muestran diferencias significativas
     en tasa de conversión — serán importantes en el modelo.
   → Los disparos bajo presión tienen menor tasa de conversión.

4. OUTLIERS
   → Disparos lejanos son eventos reales y válidos.
     No se eliminan — se conservan en el dataset completo.

5. MISSING VALUES
   → Sin valores ausentes críticos en las variables de modelización.
   → under_pressure y shot_first_time: NaN = False (imputado).

6. PRÓXIMOS PASOS (Sección 5)
   → Encoding de variables categóricas.
   → Partición train/validación/test estratificada.
   → Construcción del dataset final para modelización.
""")

# ─── Guardar dataset limpio para la siguiente sección ──────────────────────
df.to_csv('dataset_disparos_limpio.csv', index=False)
print('💾 Dataset guardado como dataset_disparos_limpio.csv')
print('   (Este archivo se usará como input en el notebook 02_preparacion_datos.ipynb)')